In [1]:
import os
import json
import subprocess
from datasets import load_dataset

# Configuration
DATASET_NAME = "bigcode/humanevalpack"
LANGUAGE_FILTER = "java"
OUTPUT_JSON_FILE = "english_to_krakatau_dataset.json"
TEMP_CLASS_NAME = "Solution"

def clean_temp_files():
    """Cleans up leftover compile files so iterations don't bleed together."""
    for ext in [".java", ".class", ".j"]:
        filename = f"{TEMP_CLASS_NAME}{ext}"
        if os.path.exists(filename):
            os.remove(filename)

print(f"Loading '{DATASET_NAME}' from Hugging Face...")
# Load the Python/Java coding benchmark dataset
data= load_dataset(DATASET_NAME, "java")
dataset = data["test"]
print(f"Dataset loaded with {len(dataset)} problems.")

d:\Study_Repos\java-dataset\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading 'bigcode/humanevalpack' from Hugging Face...


Dataset loaded with 164 problems.


In [2]:
item = dataset[1]
prompt = item.get("instruction", "Write a Java program for the following task.")
java_code = item.get("declaration", "") + "\n" + item.get("canonical_solution", "")

print(java_code)

# Wrap the raw function in a valid public class structure so javac can compile it
if "class Solution" not in java_code:
    java_code = f"public class {TEMP_CLASS_NAME} {{\n{java_code}\n}}"

clean_temp_files()

    # 1. Write the Java source code to disk
with open(f"{TEMP_CLASS_NAME}.java", "w", encoding="utf-8") as f:
    f.write(java_code)

# 2. Compile Java Source to Binary Bytecode (.class)
subprocess.run(["javac", f"{TEMP_CLASS_NAME}.java"], check=True, capture_output=True)

# 3. Disassemble the Binary into Krakatau Jasmin Text Syntax
# Adjust pathing if your script is located differently
disasm_cmd = ["python", "./Krakatau/disassemble.py", "-out", ".", f"{TEMP_CLASS_NAME}.class"]
subprocess.run(disasm_cmd, check=True, capture_output=True)
print("Finish")

import java.util.*;
import java.lang.*;

class Solution {
    public List<String> separateParenGroups(String paren_string) {

        List<String> result = new ArrayList<>();
        StringBuilder current_string = new StringBuilder();
        int current_depth = 0;

        for (char c : paren_string.toCharArray()) {
            if (c == '(') {
                current_depth += 1;
                current_string.append(c);
            } else if (c == ')') {
                current_depth -= 1;
                current_string.append(c);

                if (current_depth == 0) {
                    result.add(current_string.toString());
                    current_string.setLength(0);
                }
            }
        }
        return result;

    }
}
Finish


In [ ]:
if not os.path.exists(f"{TEMP_CLASS_NAME}.j"):
            raise FileNotFoundError("Krakatau failed to output a .j file.")
            
with open(f"{TEMP_CLASS_NAME}.j", "r", encoding="utf-8") as f:
    krakatau_syntax = f.read()
    
# 5. Sanity Test: Reassemble it using Krakatau to make sure it's valid
os.remove(f"{TEMP_CLASS_NAME}.class") # Remove old class file first
asm_cmd = ["python", "Krakatau/assemble.py", "-out", ".", f"{TEMP_CLASS_NAME}.j"]
subprocess.run(asm_cmd, check=True, capture_output=True)

# If it compiled and reassembled without throwing a ProcessError, it works perfectly!
print("   -> Success! Bytecode verified and compiled smoothly.")
clean_temp_files()


   -> Success! Bytecode verified and compiled smoothly.


In [4]:
final_dataset = []
# mini_dataset = dataset.select(range(5))
mini_dataset = dataset
for index, item in enumerate(mini_dataset):
    # We use 'declaration' or 'prompt' + 'canonical_solution' to form a complete valid Java file
    prompt = item.get("instruction", "Write a Java program for the following task.")
    java_code = item.get("declaration", "") + "\n" + item.get("canonical_solution", "")
    
    # Wrap the raw function in a valid public class structure so javac can compile it
    if "class Solution" not in java_code:
        java_code = f"public class {TEMP_CLASS_NAME} {{\n{java_code}\n}}"
        
    print(f"[{index + 1}/{len(mini_dataset)}] Processing problem ID: {item.get('task_id')}")
    clean_temp_files()

    try:
        # 1. Write the Java source code to disk
        with open(f"{TEMP_CLASS_NAME}.java", "w", encoding="utf-8") as f:
            f.write(java_code)
        
        # 2. Compile Java Source to Binary Bytecode (.class)
        subprocess.run(["javac", f"{TEMP_CLASS_NAME}.java"], check=True, capture_output=True)
        
        # 3. Disassemble the Binary into Krakatau Jasmin Text Syntax
        # Adjust pathing if your script is located differently
        disasm_cmd = ["python", "./Krakatau/disassemble.py", "-out", ".", f"{TEMP_CLASS_NAME}.class"]
        subprocess.run(disasm_cmd, check=True, capture_output=True)
        
        # 4. Verify the Assembly: Read the generated .j file text
        if not os.path.exists(f"{TEMP_CLASS_NAME}.j"):
            raise FileNotFoundError("Krakatau failed to output a .j file.")
            
        with open(f"{TEMP_CLASS_NAME}.j", "r", encoding="utf-8") as f:
            krakatau_syntax = f.read()
            
        # 5. Sanity Test: Reassemble it using Krakatau to make sure it's valid
        os.remove(f"{TEMP_CLASS_NAME}.class") # Remove old class file first
        asm_cmd = ["python", "Krakatau/assemble.py", "-out", ".", f"{TEMP_CLASS_NAME}.j"]
        subprocess.run(asm_cmd, check=True, capture_output=True)
        
        # If it compiled and reassembled without throwing a ProcessError, it works perfectly!
        print("   -> Success! Bytecode verified and compiled smoothly.")
        
        # Append pair to our training dataset list
        final_dataset.append({
            "instruction": prompt,
            "input": "",
            "output": krakatau_syntax
        })

    except subprocess.CalledProcessError as e:
        # If a problem contains broken code or compilation features Krakatau hates, skip it safely
        print(f"   -> Skipped due to compilation/assembly error.")
        continue
    except Exception as e:
        print(f"   -> Skipped due to error: {e}")
        continue

# Save everything into an Alpaca/LoRA structured JSON file
print(f"\nWriting verified dataset to {OUTPUT_JSON_FILE}...")
with open(OUTPUT_JSON_FILE, "w", encoding="utf-8") as f:
    json.dump(final_dataset, f, indent=4)
    
clean_temp_files()
print(f"Done! Successfully generated {len(final_dataset)} verified pairs.")

[1/164] Processing problem ID: Java/0
   -> Success! Bytecode verified and compiled smoothly.
[2/164] Processing problem ID: Java/1
   -> Success! Bytecode verified and compiled smoothly.
[3/164] Processing problem ID: Java/2
   -> Success! Bytecode verified and compiled smoothly.
[4/164] Processing problem ID: Java/3
   -> Success! Bytecode verified and compiled smoothly.
[5/164] Processing problem ID: Java/4
   -> Success! Bytecode verified and compiled smoothly.
[6/164] Processing problem ID: Java/5
   -> Success! Bytecode verified and compiled smoothly.
[7/164] Processing problem ID: Java/6
   -> Success! Bytecode verified and compiled smoothly.
[8/164] Processing problem ID: Java/7
   -> Success! Bytecode verified and compiled smoothly.
[9/164] Processing problem ID: Java/8
   -> Success! Bytecode verified and compiled smoothly.
[10/164] Processing problem ID: Java/9
   -> Success! Bytecode verified and compiled smoothly.
[11/164] Processing problem ID: Java/10
   -> Success! Byte